In [ ]:
import os
import base64
from pathlib import Path
from dotenv import load_dotenv

from email.mime.text import MIMEText
from google_auth_oauthlib.flow import InstalledAppFlow
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# PROJECT ROOT (one level above this script / notebook folder)
try:
    PROJECT_DIR = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_DIR = Path.cwd().parent  # Jupyter fallback

TOKEN_PATH = PROJECT_DIR / "gmail_token.json"
CREDENTIALS_PATH = PROJECT_DIR / "credentials.json"

print("Project:", PROJECT_DIR)
print("Token exists:", TOKEN_PATH.exists())
print("Credentials exists:", CREDENTIALS_PATH.exists())

# API CONFIG SETUP
load_dotenv(PROJECT_DIR / ".env")

SCOPES = ["https://www.googleapis.com/auth/gmail.send"]
TO_EMAIL = os.getenv("GMAIL_RECIPIENT")

if not TO_EMAIL:
    raise ValueError("Missing GMAIL_RECIPIENT in .env")

# AUTH
creds = None

if TOKEN_PATH.exists():
    creds = Credentials.from_authorized_user_file(
        str(TOKEN_PATH),
        SCOPES
    )

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
        TOKEN_PATH.write_text(creds.to_json())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            str(CREDENTIALS_PATH),
            SCOPES
        )
        creds = flow.run_local_server(port=0)
        TOKEN_PATH.write_text(creds.to_json())

# GMAIL SERVICE INITIALIZATION
service = build("gmail", "v1", credentials=creds)

# EMAIL CONTENT
subject = "Test Email - Gmail API"
body = """
Hey, man!

This email was sent using the Gmail API with OAuth authentication.

Best regards,

Your Python bot Assistant 😉
"""

msg = MIMEText(body)
msg["to"] = TO_EMAIL
msg["subject"] = subject

raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()

# SEND EMAIL
service.users().messages().send(
    userId="me",
    body={"raw": raw}
).execute()

print("Email sent successfully!")

Project: c:\Users\Willi\Desktop\Data Analytic Projects\Population analysis
Token exists: True
Credentials exists: True
✅ Email sent successfully!
